# AI Social Media Agent — Prompt Template Pipeline
## Abhinav Gupta | IITB Capstone 2026

---

## SECTION 1: SETUP & CONFIGURATION

In [ ]:
# Cell 1 — Install required libraries
# Run once, then comment out
# !pip install langchain langchain-anthropic langchain-openai openai anthropic -q
# !pip install python-dotenv transformers torch faiss-cpu pdfplumber chromadb -q
# !pip install xgboost nltk
print("✅ Libraries ready!")

In [ ]:
# Cell 2 — All imports in one place
import os
import pickle
import pandas as pd
import numpy as np
import re
import json
import random
import sys
import nltk
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from transformers import pipeline as hf_pipeline
from nltk.sentiment.vader import SentimentIntensityAnalyzer

print("✅ All imports loaded!")

# Prevents libomp conflicts on Mac ARM when
# torch + faiss + xgboost are all loaded together
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

print("✅ Environment variables set!")

In [ ]:
# Cell 3 — Load API keys securely from .env file
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
env_path = os.path.join(repo_root, '.env')
load_dotenv(dotenv_path=env_path)

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if ANTHROPIC_API_KEY:
    print("Anthropic key loaded:", ANTHROPIC_API_KEY[:10] + "...")
else:
    print("❌ Anthropic key NOT found — check your .env file")

if OPENAI_API_KEY:
    print("OpenAI key loaded:", OPENAI_API_KEY[:10] + "...")
else:
    print("❌ OpenAI key NOT found — check your .env file")

In [ ]:
# Cell 4 — Initialize LLM (Claude Haiku for dev, Sonnet for production)
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=4000
)

# Quick test
response = llm.invoke([HumanMessage(content="Say hello in 5 words.")])
print("LLM test:", response.content)
print("✅ LLM initialized!")

In [ ]:
# Cell 5 — Load DistilBERT sentiment model (loads once per session)
print("Loading DistilBERT sentiment model (~15 seconds)...")
sentiment_analyzer = hf_pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✅ DistilBERT loaded!")

---

## SECTION 2: PROMPT TEMPLATES

Three generations of templates to demonstrate optimization progression:
- **Level 1:** Zero-shot (no examples, no reasoning)
- **Level 2:** Few-shot (with real brand examples)
- **Level 3:** Unified (few-shot + chain-of-thought + JSON output)

Use these to show examiners how each optimization improves quality.

In [ ]:
# Cell 6 — LEVEL 1: Zero-Shot Templates (baseline — no examples, no COT)
# Used for comparison to show improvement from few-shot and COT

zeroshot_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform"],
    template="""
You are an expert social media content creator.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags
- Instagram: storytelling style, 5-10 hashtags, strong opening hook

OUTPUT FORMAT:
Return exactly 5 posts numbered like this:
1. [post text] [hashtags]
2. [post text] [hashtags]
3. [post text] [hashtags]
4. [post text] [hashtags]
5. [post text] [hashtags]

Write only the posts. No explanations.
"""
)

print("✅ Zero-shot template loaded!")

In [ ]:
# Cell 7 — Few-shot examples library
# Sources: Nike (Twitter), Adobe (LinkedIn), NatGeo (Instagram)
# Real posts from real brands — not fabricated

twitter_examples = """
EXAMPLE 1 (Athlete/Event Celebration - Nike):
"When you're this fast, you don't ask for permission. Jutta Leerdam
breaks the Olympic record in the Speed Skating 1000m and wins her
first Gold. #MilanoCortina2026 #Olympics"

EXAMPLE 2 (Product Launch - Nike):
"Mute the gallery. Introducing Nike Powerbeats Pro 2, the ultimate
fitness earbuds. The first-ever Beats x Nike collab — where premium
sound meets unbeatable stability. No advice required.
Launching March 20th at 9am PST."

EXAMPLE 3 (Motivational/Quote - Nike):
"There's no failure in sports. It's steps to success." - @Giannis_An34
Regardless of the outcome, there's always a reward ahead. #AlwaysForward
"""

linkedin_examples = """
EXAMPLE 1 (Thought Leadership - Adobe):
"Creative teams are under pressure to deliver more content across more
channels than ever before. Many are turning to AI to handle production
tasks like resizing, versioning, and early-stage ideation, so they can
spend more time refining concepts and craft. Adobe's latest research
breaks down how this shift is changing the way we do creative work."

EXAMPLE 2 (Corporate Announcement - Adobe):
"The way brands are discovered is changing fast. Today, Adobe completed
its acquisition of Semrush, expanding how businesses show up, get found,
and drive growth in an AI-first world. AI-driven traffic to U.S. retail
sites is up 269% year over year, yet most businesses still have significant
gaps in how they appear across AI surfaces."

EXAMPLE 3 (Community/Human Story - Adobe):
"My greatest inspiration came from being raised by a self-made single
mom who showed us strength and resilience. From drawing at a young age
to now utilizing Adobe tools to bridge the gap between art and business,
Alexandra Yvette continues to create and share her talents with the
world this Women's History Month."
"""

instagram_examples = """
EXAMPLE 1 (Wildlife/Action Story - NatGeo):
"This image was taken near Polán, Toledo province (Spain) last August,
from a tiny hide-out overlooking a small waterhole where lynxes
occasionally come down to drink. It was an extremely hot afternoon,
and a rabbit was very close to the water. Suddenly, a lynx appeared
silently, but the rabbit noticed it at the very last second.
Photo by @alexandrovich_yo from our @natgeoyourshot community."

EXAMPLE 2 (Landscape/Philosophical - NatGeo):
"From Punta Helbronner, 11,370 feet above sea level on the Italian
side of Mont Blanc in the Alps, my guide walks out across the snow
toward Dent du Géant, the Giant's Tooth, illuminated only by the
light of my drone during a long exposure. Photos by @reuben"

EXAMPLE 3 (Milestone/Achievement - NatGeo):
"Barcelona's Sagrada Família has reached a new milestone. With the
cross now installed atop its central Jesus tower, the basilica stands
more than 560 feet (170m) — surpassing Germany's Ulm Minster as the
world's tallest church. Photographs by Nuria Puentes, National Geographic-España"
"""

def get_examples(platform):
    if platform == "twitter":
        return twitter_examples
    elif platform == "linkedin":
        return linkedin_examples
    elif platform == "instagram":
        return instagram_examples
    else:
        return ""

print("✅ Few-shot examples library loaded!")

In [ ]:
# Cell 8 — LEVEL 2: Few-Shot Template (with real brand examples)
# Improvement over zero-shot: Claude has concrete targets to aim for

fewshot_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform", "examples"],
    template="""
You are an expert social media content creator.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags
- Instagram: storytelling style, 5-10 hashtags, strong opening hook

HIGH PERFORMING EXAMPLES FOR REFERENCE:
{examples}

Now write 5 posts matching the same quality and style as the examples above.

OUTPUT FORMAT:
Return exactly 5 posts numbered like this:
1. [post text] [hashtags]
2. [post text] [hashtags]
3. [post text] [hashtags]
4. [post text] [hashtags]
5. [post text] [hashtags]

Write only the posts. No explanations.
"""
)

print("✅ Few-shot template loaded!")

In [ ]:
# Cell 9 — LEVEL 3: Unified Template (few-shot + COT + JSON output)
# Our production template — best quality output

unified_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform", "examples", "brand_context"],
    template="""
You are an expert social media content creator specializing in brand voice alignment.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

BRAND GUIDELINES FOR {brand_name}:
{brand_context}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags, short paragraphs
- Instagram: storytelling style, strong opening hook, 5-10 hashtags, emojis throughout

HIGH PERFORMING EXAMPLES FOR REFERENCE:
{examples}

THINK STEP BY STEP BEFORE WRITING EACH VARIANT:
Step 1 - Audience: Who is the primary audience for this variant?
Step 2 - Key Message: What is the single most important message?
Step 3 - Tone Check: Does the tone match the brand and platform?
Step 4 - Hook: What opening line will stop the scroll?
Step 5 - CTA: What action should the reader take?

Make each variant structurally distinct:
- Variant 1: Lead with business impact
- Variant 2: Lead with customer benefit
- Variant 3: Lead with a bold statement or question
- Variant 4: Lead with data or proof points
- Variant 5: Lead with a human or emotional angle

IMPORTANT: Respond with ONLY a valid JSON array.
No explanations. No preamble. No markdown. Just the JSON.

Return exactly this structure:
[
  {{
    "variant_id": 1,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 2,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 3,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 4,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 5,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }}
]
"""
)

print("✅ Unified production template loaded!")

---

## SECTION 3: PIPELINE FUNCTIONS

Core functions for generating and parsing posts.

In [ ]:
# Cell 10 — JSON parser helper
def parse_json_response(response_text):
    """
    Cleans and parses Claude's JSON response
    Handles markdown code blocks if present
    """
    text = response_text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing failed: {e}")
        return None

print("✅ JSON parser ready!")

In [ ]:
# Cell 11 — Main pipeline function (with RAG integration)
sys.path.append('../../')

# Import RAG module
try:
    from modules.rag import build_index, retrieve_brand_context, is_index_built
    RAG_AVAILABLE = True
    print("✅ RAG module loaded!")
except ImportError:
    RAG_AVAILABLE = False
    print("⚠️ RAG module not found — running without brand context")


def generate_posts(brand_name, topic, tone, platform, 
                   pdf_path=None, brand_context=None):
    """
    Main pipeline — generates 5 post variants using
    unified template (few-shot + COT + JSON + RAG)

    Args:
        brand_name (str): e.g. "Adobe"
        topic (str): e.g. "Adobe's acquisition of Semrush"
        tone (str): e.g. "professional, forward looking"
        platform (str): "twitter", "linkedin", "instagram"
        pdf_path (str): optional path to brand voice PDF
        brand_context (str): optional pre-retrieved brand context

    Returns:
        list: 5 structured post variants as Python list of dicts
    """
    examples = get_examples(platform)

    # ── Step 1: Get brand context from RAG ──
    if brand_context:
        # Use pre-provided context (e.g. from Streamlit upload)
        context = brand_context
        print(f"✅ Using provided brand context")

    elif pdf_path and RAG_AVAILABLE:
        # Build index from PDF if not already built
        if not is_index_built(brand_name):
            build_index(pdf_path, brand_name)
        else:
            print(f"✅ Using existing RAG index for {brand_name}")

        # Retrieve relevant chunks
        context = retrieve_brand_context(
            brand_name=brand_name,
            topic=topic,
            platform=platform,
            tone=tone
        )
        print(f"✅ Retrieved brand context ({len(context)} chars)")

    else:
        # No PDF provided — generate without brand context
        context = (f"Write in a {tone} tone appropriate for "
                  f"{brand_name} on {platform}.")
        print("⚠️ No brand PDF provided — using default context")

    # ── Step 2: Assemble and fill prompt ──
    filled_prompt = unified_template.format(
        brand_name=brand_name,
        topic=topic,
        tone=tone,
        platform=platform,
        examples=examples,
        brand_context=context
    )

    # ── Step 3: Call Claude ──
    response = llm.invoke([HumanMessage(content=filled_prompt)])

    # ── Step 4: Parse and return JSON ──
    return parse_json_response(response.content)


print("✅ generate_posts() with RAG ready!")

---

## SECTION 4: FEATURE EXTRACTION & SCORING

Extracting engagement features from each generated post and
scoring them using Manas's trained XGBoost model.

- **Cell 12:** DistilBERT sentiment (for display/analysis)
- **Cell 13:** Feature extractor — extracts 9 features per post
- **Cell 14:** XGBoost scorer — loads Manas's real model
- **Cell 15:** Optimizer — ranks 5 variants by predicted engagement

NOTE: VADER sentiment is used for scoring (matches Manas's training data).
DistilBERT is kept separately for display purposes only.

In [ ]:
# Cell 12 — Sentiment scorer using DistilBERT
def get_sentiment_score(text):
    """
    Returns sentiment score from -1 to 1
    Positive posts score closer to +1
    Negative posts score closer to -1
    Uses DistilBERT for contextual understanding
    """
    result = sentiment_analyzer(text[:512])[0]
    score = result['score']
    if result['label'] == 'POSITIVE':
        return round(score, 4)
    else:
        return round(-score, 4)

# Test it
test_score = get_sentiment_score("Adobe's acquisition of Semrush strengthens our market position!")
print(f"Test sentiment score: {test_score}")
print("✅ Sentiment scorer ready!")

In [ ]:
# Cell 13 — Feature extractor (corrected to match Manas's model schema)


# Download VADER if needed
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

# Initialize VADER — matches what Manas used to train XGBoost
vader_analyzer = SentimentIntensityAnalyzer()

def get_vader_score(text):
    """
    VADER sentiment for XGBoost feature.
    NOTE: We use VADER (not DistilBERT) because Manas 
    trained XGBoost on VADER scores. Feature name must be
    new_sentiment_score to match Manas's model schema.
    """
    return round(vader_analyzer.polarity_scores(text)['compound'], 4)

def extract_features(variant, platform):
    """
    Extracts features matching Manas's predict_engagement_kaggle() schema exactly.
    
    Args:
        variant: single post dict from generate_posts()
        platform: "twitter", "linkedin", "instagram"
    
    Returns:
        dict of features matching Manas's XGBoost model schema
    """
    post_text = variant['post_text']
    hashtags = variant['hashtags']
    posting_time = variant.get('suggested_posting_time', '9:00 AM')

    # Feature 1 — Caption length
    caption_length = len(post_text)

    # Feature 2 — Hashtag count
    hashtag_count = len(hashtags)

    # Feature 3 — VADER sentiment (named new_sentiment_score to match Manas)
    new_sentiment_score = get_vader_score(post_text)

    # Feature 4 — Has CTA
    cta_keywords = [
        'link', 'register', 'learn more', 'sign up', 'click here',
        'visit', 'download', 'bio', 'get started', 'click',
        'discover', 'explore', 'try', 'get', 'join', 'shop',
        'buy', 'check out', 'read', 'watch', 'follow', 'share'
    ]
    has_cta = int(any(kw in post_text.lower() for kw in cta_keywords))

    # Feature 5 — Platform encoded (named platform_encoded to match Manas)
    platform_map = {"twitter": 0, "linkedin": 1, "instagram": 2}
    platform_encoded = platform_map.get(platform, 0)

    # Feature 6 — Hour posted (required by Manas's model)
    # Default to 9am — hour is used by model but NOT for ranking content quality
    hour_match = re.search(r'(\d+):\d+\s*(AM|PM)', posting_time)
    if hour_match:
        hour = int(hour_match.group(1))
        if hour_match.group(2) == 'PM' and hour != 12:
            hour += 12
        elif hour_match.group(2) == 'AM' and hour == 12:
            hour = 0
    else:
        hour = 9

    # Additional features (not used by Manas's model but useful for future)
    word_count = len(post_text.split())
    has_question = int('?' in post_text)
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF" "]+",
        flags=re.UNICODE
    )
    has_emoji = int(bool(emoji_pattern.search(post_text)))

    return {
        # Core features — match Manas's model exactly
        "caption_length":      caption_length,
        "hashtag_count":       hashtag_count,
        "new_sentiment_score": new_sentiment_score,  # VADER, matches Manas
        "has_cta":             has_cta,
        "platform_encoded":    platform_encoded,     # matches Manas
        "hour_posted":         hour,                 # required by Manas's model

        # Additional features for display/future use
        "word_count":          word_count,
        "has_question":        has_question,
        "has_emoji":           has_emoji
    }

print("✅ Feature extractor ready!")


In [ ]:
# Cell 14 — Engagement Scorer (Manas's real XGBoost model)


# Load Manas's trained model and column schema
MODEL_PATH = os.path.join(os.getcwd(), '..', '..', 'models', 'xgboost_model.pkl')
COLS_PATH  = os.path.join(os.getcwd(), '..', '..', 'models', 'xgboost_columns.pkl')

try:
    with open(MODEL_PATH, 'rb') as f:
        xgb_model = pickle.load(f)
    with open(COLS_PATH, 'rb') as f:
        train_columns = pickle.load(f)
    SCORING_AVAILABLE = True
    print("✅ Manas's XGBoost model loaded!")
    print(f"   Features: {len(train_columns)} columns")
except FileNotFoundError:
    SCORING_AVAILABLE = False
    print("⚠️ Model files not found in models/ folder")
    print("   Posts will be generated but not scored")
except Exception as e:
    SCORING_AVAILABLE = False
    print(f"⚠️ Could not load model: {e}")
    print("   Posts will be generated but not scored")


def predict_engagement(features_dict: dict) -> float:
    """
    Predicts engagement score using Manas's trained XGBoost model.
    Raises exception if model not available — no fake fallback.

    Args:
        features_dict: must contain:
            - caption_length (int)
            - hashtag_count (int)
            - new_sentiment_score (float, VADER -1 to 1)
            - has_cta (int, 0 or 1)
            - platform_encoded (int, 0=Twitter 1=LinkedIn 2=Instagram)
            - hour_posted (int, 0-23)

    Returns:
        float: predicted engagement score (higher = better)

    Raises:
        RuntimeError: if model is not available
    """
    if not SCORING_AVAILABLE:
        raise RuntimeError(
            "Engagement model not available. "
            "Ensure xgboost_model.pkl and xgboost_columns.pkl "
            "are in the models/ folder."
        )

    # Extract only the 6 features Manas's model needs
    model_features = {
        'caption_length':      features_dict['caption_length'],
        'hashtag_count':       features_dict['hashtag_count'],
        'new_sentiment_score': features_dict['new_sentiment_score'],
        'has_cta':             features_dict['has_cta'],
        'platform_encoded':    features_dict['platform_encoded'],
        'hour_posted':         features_dict['hour_posted']
    }

    # Build input DataFrame
    input_df = pd.DataFrame([model_features])

    # One-hot encode categorical features to match training schema
    input_df['hour_posted']     = input_df['hour_posted'].astype('category')
    input_df['platform_encoded'] = input_df['platform_encoded'].astype('category')
    processed = pd.get_dummies(
        input_df,
        columns=['hour_posted', 'platform_encoded'],
        drop_first=True
    )

    # Align columns exactly with training data
    processed = processed.reindex(columns=train_columns, fill_value=0)

    # Predict and inverse transform from log scale
    log_pred = xgb_model.predict(processed)[0]
    score = float(np.expm1(log_pred))
    return max(0.0, round(score, 4))

print("✅ Engagement scorer ready!")
print(f"   Status: {'Real XGBoost model active' if SCORING_AVAILABLE else 'Scoring unavailable — model missing'}")


In [ ]:
# Cell 15 — Optimizer (clean failure handling, no mock)
def optimize_variants(variants, platform):
    """
    Scores and ranks 5 post variants by predicted engagement.
    
    If scoring fails — returns all variants unranked with
    scoring_failed=True so the UI can handle it gracefully.
    Users can still read and select any post they prefer.
    
    NOTE: All variants scored at neutral hour=9am so posting
    time does not influence content quality ranking.

    Args:
        variants: list of 5 post dicts from generate_posts()
        platform: "twitter", "linkedin", "instagram"

    Returns:
        ranked_variants: list — scored and sorted if successful,
                         original order with scoring_failed flag if not
        scores: list of scores, or empty list if scoring failed
        scoring_failed: bool — True if scoring could not be completed
    """
    scores = []
    scoring_failed = False

    try:
        print("Extracting features and scoring variants...")

        for i, variant in enumerate(variants):
            features = extract_features(variant, platform)

            # Fix hour to neutral value — posting time is a recommendation
            # not a content quality signal
            features['hour_posted'] = 9

            score = predict_engagement(features)
            scores.append(score)
            variant['predicted_score'] = score
            variant['is_recommended'] = False
            variant['scoring_failed'] = False
            print(f"  Variant {i+1}: score = {score}")

        # Sort descending by predicted score
        ranked = sorted(
            variants,
            key=lambda x: x['predicted_score'],
            reverse=True
        )

        # Flag top variant as recommended
        ranked[0]['is_recommended'] = True

        print(f"\n⭐ Recommended: Variant {ranked[0]['variant_id']} "
              f"(score: {ranked[0]['predicted_score']})")

        return ranked, scores, False

    except RuntimeError as e:
        # Model not available — return posts unscored
        print(f"\n⚠️ Scoring unavailable: {e}")
        print("   Returning all posts unscored.")
        print("   You can still read and select the post you prefer.")

        for variant in variants:
            variant['predicted_score'] = None
            variant['is_recommended'] = False
            variant['scoring_failed'] = True

        return variants, [], True

    except Exception as e:
        # Unexpected error — return posts unscored
        print(f"\n⚠️ Scoring failed unexpectedly: {e}")
        print("   Returning all posts unscored.")
        print("   You can still read and select the post you prefer.")

        for variant in variants:
            variant['predicted_score'] = None
            variant['is_recommended'] = False
            variant['scoring_failed'] = True

        return variants, [], True

print("✅ Optimizer ready!")


---

## SECTION 5: COMPARISON DEMO

**For examiners — shows how each optimization improves quality.**

Run these 3 cells with the same topic and platform to compare:
- Cell 16: Zero-shot output (baseline)
- Cell 17: Few-shot output (with examples)
- Cell 18: Unified output (few-shot + COT + JSON + scoring)

In [ ]:
# Cell 16 — COMPARISON: Zero-Shot Output (baseline)
# Same topic and platform for fair comparison

DEMO_BRAND = "Adobe"
DEMO_TOPIC = "Adobe's acquisition of Semrush"
DEMO_TONE = "professional, strategic, forward looking"
DEMO_PLATFORM = "linkedin"

print("=" * 60)
print("LEVEL 1: ZERO-SHOT (no examples, no reasoning)")
print("=" * 60)

zeroshot_filled = zeroshot_template.format(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM
)

zeroshot_response = llm.invoke([HumanMessage(content=zeroshot_filled)])
print(zeroshot_response.content)

In [ ]:
# Cell 17 — COMPARISON: Few-Shot Output
# Same topic — now with real brand examples

print("=" * 60)
print("LEVEL 2: FEW-SHOT (with Nike/Adobe/NatGeo examples)")
print("=" * 60)

fewshot_filled = fewshot_template.format(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM,
    examples=get_examples(DEMO_PLATFORM)
)

fewshot_response = llm.invoke([HumanMessage(content=fewshot_filled)])
print(fewshot_response.content)

In [ ]:
# Cell 18 — COMPARISON: Unified Output (few-shot + COT + JSON + scoring)
# Same topic — now with full pipeline including optimization

print("=" * 60)
print("LEVEL 3: UNIFIED (few-shot + COT + JSON + optimizer)")
print("=" * 60)

# Generate 5 variants using full pipeline
variants = generate_posts(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM
)

if variants:
    ranked_variants, scores, scoring_failed = optimize_variants(
        variants, DEMO_PLATFORM
    )

    print("\n" + "=" * 60)
    if scoring_failed:
        print("GENERATED POSTS — SCORING UNAVAILABLE")
        print("Select the post you prefer:")
    else:
        print("OPTIMIZATION RESULTS — RANKED BY PREDICTED ENGAGEMENT")
    print("=" * 60)

    for i, variant in enumerate(ranked_variants):
        if scoring_failed:
            label = f"Post {i+1}"
            score_display = "Score: N/A (model unavailable)"
        else:
            recommended = "⭐ RECOMMENDED" if variant['is_recommended'] else ""
            label = f"Rank {i+1} {recommended}"
            score_display = f"Score: {variant['predicted_score']}"

        print(f"\n{label}")
        print(f"{score_display}")
        print(f"Reasoning: {variant.get('reasoning', 'N/A')[:80]}...")
        print(f"Preview: {variant['post_text'][:100]}...")
        print(f"Hashtags: {' '.join(variant['hashtags'])}")
        print(f"Suggested posting time: {variant['suggested_posting_time']}")

    print("\n" + "=" * 60)
    if scoring_failed:
        print("ALL POSTS — CHOOSE YOUR PREFERRED:")
    else:
        print("RECOMMENDED POST — FULL TEXT:")
    print("=" * 60)
    print(ranked_variants[0]['post_text'])
    print(f"\nHashtags: {' '.join(ranked_variants[0]['hashtags'])}")
    if not scoring_failed:
        print(f"Predicted engagement score: {ranked_variants[0]['predicted_score']}")
    print(f"Suggested posting time: {ranked_variants[0]['suggested_posting_time']}")
else:
    print("❌ Generation failed")


---

## SECTION 6: FULL PIPELINE TEST

Test the complete pipeline across all 3 platforms and multiple brands.

In [ ]:
# Cell 19 — Test across all 3 platforms
test_brand = "Adobe"
test_topic = "Adobe's acquisition of Semrush"
test_tone = "professional, strategic, forward looking"

print("FULL PIPELINE TEST — ALL 3 PLATFORMS")
print("=" * 60)

all_results = {}

for platform in ["twitter", "linkedin", "instagram"]:
    print(f"\nGenerating for {platform.upper()}...")

    variants = generate_posts(test_brand, test_topic, test_tone, platform)

    if variants:
        ranked, scores, scoring_failed = optimize_variants(variants, platform)
        all_results[platform] = ranked

        if scoring_failed:
            print(f"✅ {platform.upper()} — {len(variants)} variants generated")
            print(f"   ⚠️ Scoring unavailable — showing first post")
            print(f"   Preview: {ranked[0]['post_text'][:80]}...")
        else:
            print(f"✅ {platform.upper()} — {len(variants)} variants generated")
            print(f"   Best score: {ranked[0]['predicted_score']}")
            print(f"   Recommended: {ranked[0]['post_text'][:80]}...")
    else:
        print(f"❌ {platform.upper()} — generation failed")

print("\n" + "=" * 60)
print(f"SUMMARY: {len(all_results)}/3 platforms successful")


In [ ]:
# Cell 20 — Test with different brand
# Change brand_name and tone to see different outputs

PDF_PATH = "../../data/brand_guidelines/Adidas_Brand_Voice.pdf"

variants_adidas = generate_posts(
    brand_name="Adidas",
    topic="launching new running shoe",
    tone="bold, performance-driven, athlete-focused",
    platform="instagram",
    pdf_path=PDF_PATH
)

if variants_adidas:
    ranked_adidas, scores_adidas, scoring_failed = optimize_variants(
        variants_adidas, "instagram"
    )

    print("ADIDAS — INSTAGRAM")
    print("=" * 60)

    if scoring_failed:
        print("⚠️ Scoring unavailable — showing all posts unranked")
        print("Select the post you prefer:\n")
        for i, v in enumerate(ranked_adidas):
            print(f"Post {i+1}:")
            print(f"{v['post_text'][:150]}...")
            print(f"Hashtags: {' '.join(v['hashtags'])}")
            print()
    else:
        print(f"⭐ RECOMMENDED POST (Score: {ranked_adidas[0]['predicted_score']})")
        print("=" * 60)
        print(ranked_adidas[0]['post_text'])
        print(f"\nHashtags: {' '.join(ranked_adidas[0]['hashtags'])}")
        print(f"Suggested posting time: {ranked_adidas[0]['suggested_posting_time']}")
        print(f"Score: {ranked_adidas[0]['predicted_score']}")
else:
    print("❌ Generation failed")


---

## SECTION 7: INTEGRATION STATUS

### ✅ Completed Integrations

**Manas's XGBoost Model**
Model loaded directly from `models/xgboost_model.pkl`.
Feature schema matches Manas's training data exactly:
- `caption_length`, `hashtag_count`, `new_sentiment_score` (VADER)
- `has_cta`, `platform_encoded`, `hour_posted`

**Shreyas's RAG Module**
`modules/rag.py` integrated into `generate_posts()`.
Brand voice PDF → FAISS index → top-k chunks → injected into prompt.

**Production Modules**
All functions extracted into importable modules:
- `modules/pipeline.py`  → generate_posts(), optimize_variants()
- `modules/predictor.py` → predict_engagement()
- `modules/rag.py`       → build_index(), retrieve_brand_context()

### 🔄 Pending Integrations

**Trending Topics (Shreyas)**
When `modules/trending.py` is ready:
```python
# Add to generate_posts() in pipeline.py:
from modules.trending import get_trending_topics
trending = get_trending_topics(topic)
# Add {trending_topics} field to UNIFIED_TEMPLATE
```

**Connect app.py (Aditya)**
Aditya's Streamlit app imports from modules:
```python
from modules.pipeline import generate_posts, optimize_variants
from modules.rag import build_index, retrieve_brand_context
```

### JSON Schema for Aditya's UI
Each variant in generate_posts() output:
```json
{
  "variant_id": 1,
  "reasoning": "Claude's thinking for this variant",
  "post_text": "the actual post content",
  "hashtags": ["#hashtag1", "#hashtag2"],
  "tone": "tone used",
  "suggested_posting_time": "Tuesday 9:00 AM ET",
  "platform": "linkedin",
  "predicted_score": 245.67,
  "is_recommended": true,
  "scoring_failed": false
}
```

NOTE: predicted_score is a raw engagement number, not 0 to 1.
Higher = better. Used for ranking only.